# Airborne Tracking Dataset Analysis

Interactive analysis of the COCO-format airborne tracking dataset.

**Datasets analyzed:**
- `airborne_tracking_coco.json` (full dataset — loaded as the `full` split)
- `train.json`, `eval.json`, `test.json` (split datasets)

All analyses are performed identically across selected splits. Edit `SELECTED_SPLITS` to choose which splits to plot.

In [ ]:
import json
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
# from plotly.subplots import make_subplots
from pathlib import Path
# from collections import Counter

BASE_DIR = "/mnt/Pool_IA/home/vision/mpf5/DAA/items/sw_ai/items/sw_ai_detection/items/"

IMAGES_DIR = "/mnt/Pool_IA/IA_Dataset/datasets/airborne-obj-detection-dataset/airborne_cropped_images/"
FULL_DATASET = Path(BASE_DIR) / "data/03_size_metadata/airborne_tracking_coco.json"
SPLITS_FOLDER = Path(BASE_DIR) / "data/06_split/"

# ── Select which splits to analyze ──────────────────────────────────
# Comment out or remove entries you don't want.
SELECTED_SPLITS = [
    # "full",
    "train",
    "eval",
    "test"
    ]
# ────────────────────────────────────────────────────────────────────

def load_coco_json(path: Path) -> dict:
    with open(path) as f:
        return json.load(f)

def coco_to_dataframes(coco: dict, split_name: str = "all") -> tuple[pd.DataFrame, pd.DataFrame]:
    """Convert COCO dict to annotations and images DataFrames."""
    cat_map = {c["id"]: c["name"] for c in coco["categories"]}

    df_ann = pd.DataFrame(coco["annotations"])
    df_ann["class"] = df_ann["category_id"].map(cat_map)
    df_ann["bbox_x"] = df_ann["bbox"].apply(lambda b: b[0])
    df_ann["bbox_y"] = df_ann["bbox"].apply(lambda b: b[1])
    df_ann["bbox_w"] = df_ann["bbox"].apply(lambda b: b[2])
    df_ann["bbox_h"] = df_ann["bbox"].apply(lambda b: b[3])
    df_ann["aspect_ratio"] = df_ann["bbox_w"] / df_ann["bbox_h"].replace(0, np.nan)
    df_ann["split"] = split_name

    horizon_map = {-1: "unknown", 0: "below", 1: "above"}
    if "is_above_horizon" in df_ann.columns:
        df_ann["horizon"] = df_ann["is_above_horizon"].map(horizon_map)

    df_img = pd.DataFrame(coco["images"])
    df_img["split"] = split_name

    return df_ann, df_img

# Load all datasets as splits
all_ann = []
all_img = []

if "full" in SELECTED_SPLITS and FULL_DATASET.exists():
    coco = load_coco_json(FULL_DATASET)
    df_a, df_i = coco_to_dataframes(coco, "full")
    all_ann.append(df_a)
    all_img.append(df_i)

for split_name in ["train", "eval", "test"]:
    if split_name in SELECTED_SPLITS:
        path = SPLITS_FOLDER / f"{split_name}.json"
        if path.exists():
            coco = load_coco_json(path)
            df_a, df_i = coco_to_dataframes(coco, split_name)
            all_ann.append(df_a)
            all_img.append(df_i)

df_ann = pd.concat(all_ann, ignore_index=True) if all_ann else pd.DataFrame()
df_img = pd.concat(all_img, ignore_index=True) if all_img else pd.DataFrame()


In [ ]:
all_ann

## 1. Annotation Count per Class — by Split

In [ ]:
split_class_counts = df_ann.groupby(["split", "class", "size_category"]).size().reset_index(name="count")
split_class_counts["split_class"] = split_class_counts["split"] + " / " + split_class_counts["class"]

fig = px.bar(split_class_counts, x="split_class", y="count", color="size_category",
             barmode="stack", text_auto=True,
             title="Annotation Count per Class — by Split",
             category_orders={"size_category": ["small", "medium", "large"]})
fig.show()

## 2. Images per Class — by Split

In [ ]:
images_per_class = df_ann.groupby(["split", "class", "size_category"])["image_id"].nunique().reset_index(name="num_images")

# fig = px.bar(images_per_class, x="class", y="num_images", color="split",
#              barmode="group", text_auto=True,
#              title="Number of Images Containing Each Class — by Split",
#              category_orders={"split": SELECTED_SPLITS, "size_category": ["small", "medium", "large"]})
fig = px.bar(images_per_class, x=["size_category"], y="num_images", color="split",
             text_auto=True,
             title="Number of Images Containing Each Class — by Split and Size Category",
             category_orders={"split": SELECTED_SPLITS, "size_category": ["small", "medium", "large"]})
fig.update_layout(xaxis_title="Class and Size Category", yaxis_title="Unique Images")
fig.show()

## 3. Distribution of Annotations per Image — by Split

In [ ]:
ann_per_image = df_ann.groupby(["split", "image_id"]).size().reset_index(name="num_annotations")

fig = px.histogram(ann_per_image, x="num_annotations", color="split",
                   facet_col="split", nbins=10,
                   title="Distribution of Annotations per Image — by Split",
                   category_orders={"split": SELECTED_SPLITS},
                   )
fig.update_layout(xaxis_title="Annotations per Image", yaxis_title="Image Count",
                  height=400)
fig.show()

for s in SELECTED_SPLITS:
    sub_ann = df_ann[df_ann["split"] == s]
    sub_img = df_img[df_img["split"] == s]
    api = sub_ann.groupby("image_id").size()
    empty = len(set(sub_img["id"]) - set(sub_ann["image_id"].unique()))
    print(f"[{s}] images with 0 annotations: {empty}/{len(sub_img)} "
          f"({100*empty/len(sub_img):.1f}%) | "
          f"ann/img — mean: {api.mean():.2f}, median: {api.median():.0f}, max: {api.max()}")

## 4. Horizon Position per Class — by Split

In [ ]:
fig = px.histogram(df_ann, x="class", color="horizon",
                   facet_col="split", barmode="group",
                   title="Horizon Position per Class — by Split",
                   category_orders={"split": SELECTED_SPLITS,
                                    "horizon": ["above", "below", "unknown"]},
                   color_discrete_map={"above": "#636EFA", "below": "#EF553B", "unknown": "#AB63FA"})
fig.update_layout(height=400)
fig.show()

## 5. Area by Horizon Position — by Split

In [ ]:
fig = px.violin(df_ann, x="horizon", y="area", color="split",
                box=True, points="outliers",
             facet_col="split", log_y=True,
             title="Area by Horizon Position — by Split",
             category_orders={"split": SELECTED_SPLITS,
                              "horizon": ["above", "below", "unknown"]})
fig.update_layout(height=400)
fig.show()

## 6. Area vs Distance — by Split (marker size = exact duplicate count)

Includes unknown distance (-1). Marker size encodes how many annotations share the exact same (range_m, area) pair.

In [ ]:
# Group by exact (range_m, area, split, size_category) — no rounding
area_dist_counts = (df_ann.loc[df_ann["range_m"] > 0].groupby(["split", "range_m", "area", "class"])
                    .size().reset_index(name="count"))

fig = px.scatter(area_dist_counts, x="range_m", y="area",
                 color="class", 
                 facet_col="split",
                 category_orders={"split": SELECTED_SPLITS},
                 color_discrete_map={"small": "#00CC96", "medium": "#FFA15A", "large": "#EF553B"},
                #  color_marker={"small": "#00CC96", "medium": "#FFA15A", "large": "#EF553B"},
                 opacity=0.5, log_y=True,
                 title="Area vs Distance — by Split (marker size = count of exact duplicates)",
                 hover_data=["count"])
fig.update_layout(height=450)
fig.show()

for s in SELECTED_SPLITS:
    sub = df_ann[df_ann["split"] == s]
    n_valid = (sub["range_m"] > 0).sum()
    n_unknown = (sub["range_m"] < 0).sum()
    print(f"[{s}] valid range: {n_valid:,} | unknown (-1): {n_unknown:,} / {len(sub):,}")

print(df_ann["size_category"].value_counts())

## 7. Height vs Width — by Split (marker size = exact duplicate count)

In [ ]:
# Group by exact (bbox_w, bbox_h, split, class) — no rounding
hw_counts = (df_ann.groupby(["bbox_w", "bbox_h"])
             .size()).reset_index(name="count")

fig = px.scatter(hw_counts, x="bbox_w", y="bbox_h", size="count",
                 opacity=0.5,
                 title="Height vs Width — by Split (marker size = count of exact duplicates)",
                 category_orders={"split": SELECTED_SPLITS},
                 hover_data=["count"])
fig.update_layout(height=450)
fig.update_traces(marker=dict(sizemin=6)) 
fig.show()

In [ ]:

fig = px.scatter(hw_counts.loc[hw_counts["count"] > 1], x="bbox_w", y="bbox_h", size="count", color="count",
                 opacity=0.5,
                 title="Height vs Width — by Split (marker size = count of exact duplicates)",
                 category_orders={"split": SELECTED_SPLITS},
                 hover_data=["count"])
fig.update_layout(height=450)
fig.update_traces(marker=dict(sizemin=6)) 

fig.show()

In [ ]:
(df_ann.groupby(["bbox_w", "bbox_h"]).size()).rename("count")

---
# Additional Analysis for Training Insights

## 9. Bbox Center Heatmap (Spatial Distribution) — by Split

Shows where objects appear in the image frame. Reveals biases (e.g., objects mostly in center or horizon line).

In [ ]:
# Compute bbox centers as fraction of image size
df_centers = df_ann.merge(df_img[["id", "width", "height", "split"]],
                          left_on=["image_id", "split"], right_on=["id", "split"],
                          suffixes=("", "_img"))
df_centers["cx_norm"] = (df_centers["bbox_x"] + df_centers["bbox_w"] / 2) / 960
df_centers["cy_norm"] = (df_centers["bbox_y"] + df_centers["bbox_h"] / 2) / 960

fig = px.density_heatmap(df_centers, x="cx_norm", y="cy_norm",
                         facet_col="split", nbinsx=50, nbinsy=50,
                         title="Bbox Center Heatmap (Normalized) — by Split",
                         category_orders={"split": SELECTED_SPLITS},
                         color_continuous_scale="Viridis")
fig.update_yaxes(autorange="reversed")
fig.update_layout(xaxis_title="X center (normalized)", yaxis_title="Y center (normalized)",
                  height=450)
fig.show()

## 10. Distance Distribution per Class — by Split (valid range)

Understanding range distribution helps decide if distance-aware augmentations or multi-scale training is needed.

In [ ]:
# Violin for valid ranges
fig = px.violin(df_ann[df_ann["range_m"] > 0], x="class", y="range_m", color="class",
                box=True, points="outliers",
                facet_col="split",
                category_orders={"split": SELECTED_SPLITS},
                title="Distance Distribution per Class — by Split (valid range)")
fig.update_layout(xaxis_title="Class", yaxis_title="Distance (m)", height=450)
fig.show()


## 11. Valid vs Unknown Range Count per Class — by Split

Bar chart showing the count of unknown (-1) vs valid (>0) range annotations per class.

In [ ]:
df_ann["range_category"] = df_ann["range_m"].apply(lambda x: "unknown (-1)" if x < 0 else "valid (>0)")

range_cat_counts = df_ann.groupby(["split", "class", "range_category"]).size().reset_index(name="count")

fig = px.bar(range_cat_counts, x="class", y="count", color="range_category",
             facet_col="split", barmode="group", text_auto=True,
             title="Valid vs Unknown Range Count per Class — by Split",
             category_orders={"split": SELECTED_SPLITS,
                              "range_category": ["valid (>0)", "unknown (-1)"]},
             color_discrete_map={"valid (>0)": "#636EFA", "unknown (-1)": "#AB63FA"})
fig.update_layout(height=450)
fig.show()